In [43]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from datetime import datetime

import geopandas as gpd
import quackosm as qosm

import time

In [44]:
holiday_list = pd.read_csv("data/holiday_list.csv")
outlet_cordinates = pd.read_csv("data/outlet_coordinates.csv")
outlet_master = pd.read_csv("data/outlet_master.csv")
transaction_history = pd.read_csv("data/transactions_history_final.csv")
distributor_seasonality_details = pd.read_csv("data/distributor_seasonality_details.csv")

In [45]:
# create bronze layer

In [46]:
# create lake house dirs
dirs: list[str] = ["lakehouse/bronze", "lakehouse/silver", "lakehouse/quarantine", "lakehouse/gold"]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)

In [47]:
# read files in data (raw data)\
source_file_paths: list[str] = [f"data/{path}" for path in os.listdir("data")]
source_file_paths

['data/distributor_seasonality_details.csv',
 'data/holiday_list.csv',
 'data/outlet_coordinates.csv',
 'data/outlet_master.csv',
 'data/transactions_history_final.csv']

In [48]:
LAKE_HOUSE_BRONZE_PATH = "lakehouse/bronze"
for path in source_file_paths:
    df = pd.read_csv(path)
    file_name, _ = path.rsplit(".")
    file_name = file_name.rsplit("/")[-1]
    # add meta data
    df["_ingested_at"] = pd.to_datetime(datetime.now())
    df["_source_file"] = path

    df.to_parquet(Path(f"{LAKE_HOUSE_BRONZE_PATH}/{file_name}.parquet"),compression="zstd", index=False)


In [49]:
def qurantine(df: pd.DataFrame, dataset_name: str, reason: str):
    if df.empty:
        return
    QURANTINE_PATH = "lakehouse/quarantine"
    df = df.copy()
    df['_failure_reason'] = reason
    df['_source_dataset'] = dataset_name
    path = Path(f"{QURANTINE_PATH}/{dataset_name}_{reason}.parquet")
    df.to_parquet(path, compression="zstd", index=False)

# !TODO: add success msg via print or python loggin

In [50]:
# check duplicates
def check_duplicates(df: pd.DataFrame, columns, dataset_name) -> pd.DataFrame:
    duplicates = df[df.duplicated(subset=columns, keep=False)]
    qurantine(duplicates, dataset_name, "duplicated_record")
    return df.drop_duplicates(subset=columns, keep='first')

In [51]:
# null check
def check_nulls(df: pd.DataFrame, columns, dataset_name) -> pd.DataFrame:
    mask = df[columns].isnull().any(axis=1)
    qurantine(df[mask], dataset_name, "null_in_mandatory_field")
    return df[~mask]

In [52]:
# key integrity check
def check_referential_integrity(df: pd.DataFrame, fk_col, ref_df: pd.DataFrame, ref_col, dataset_name):
    valid_keys = set(ref_df[ref_col])
    mask = ~df[fk_col].isin(valid_keys)
    qurantine(df[mask], dataset_name, f"invalid_fk_{fk_col}")
    return df[~mask]

In [53]:
def check_value_range(df: pd.DataFrame, col, min_val: int|float, max_val: int|float, dataset_name):
    mask = (df[col] < min_val) | (df[col] > max_val)
    qurantine(df[mask], dataset_name, f"{col}_out_of_range")
    return df[~mask]

In [54]:
def check_format(df: pd.DataFrame, col, dtype: np.dtype, dataset_name):
    try:
        df[col] = df[col].astype(dtype)
        return df
    except Exception:
        mask = pd.to_numeric(df[col], errors='coerce').isnull()
        qurantine(df[mask], dataset_name, f"{col}_type_mismatch")
        return df[~mask]

In [55]:
def check_geo_bounds(df, latitude, longitude, dataset_name):

    lat = df[latitude]
    lon = df[longitude]

    lat_valid = (lat >= -90) & (lat <= 90)
    lon_valid = (lon >= -180) & (lon <= 180)

    not_zero_zero = ~((lat == 0) & (lon == 0))

    valid_mask = lat_valid & lon_valid & not_zero_zero

    invalid_mask = ~valid_mask

    if invalid_mask.any():
        qurantine(df[invalid_mask], dataset_name, f"{latitude}_{longitude}_geo_out_of_bounds")

    return df[valid_mask]

In [56]:
def check_datetime_format(df, column, dataset_name, date_format=None):
    try:
        if date_format:
            df[column] = pd.to_datetime(df[column], format=date_format)
        else:
            df[column] = pd.to_datetime(df[column])
        return df
    except Exception:
        if date_format:
            converted = pd.to_datetime(df[column], format=date_format, errors='coerce')
        else:
            converted = pd.to_datetime(df[column], errors='coerce')

        invalid_mask = converted.isna()
        qurantine(df[invalid_mask], dataset_name, f"{column}_type_mismatch")

        valid_df = df[~invalid_mask].copy()
        valid_df[column] = converted[~invalid_mask]
        return valid_df

In [57]:
#silver layer


In [58]:
def save_sliver(df: pd.DataFrame, dataset_name: str):
    df.to_parquet(Path(f"{LAKE_HOUSE_SILVER_PATH}/{dataset_name}.parquet"), compression="zstd", index=False)

In [59]:
LAKE_HOUSE_SILVER_PATH = "lakehouse/silver"
os.listdir('lakehouse/bronze')

['distributor_seasonality_details.parquet',
 'holiday_list.parquet',
 'outlet_coordinates.parquet',
 'outlet_master.parquet',
 'transactions_history_final.parquet']

In [60]:
# holidaylist
df = pd.read_parquet('lakehouse/bronze/holiday_list.parquet')
dataset_name = 'holiday_list'
df = check_nulls(df, df.columns, dataset_name)
df = check_duplicates(df, df.columns, dataset_name)
df = check_datetime_format(df, 'Date', dataset_name=dataset_name)
df = check_format(df, 'Holiday_Type', 'category', dataset_name)
save_sliver(df, dataset_name)
holiday_list = df

In [61]:
tmp = pd.read_parquet('lakehouse/silver/holiday_list.parquet')
tmp.info()

<class 'pandas.DataFrame'>
RangeIndex: 256 entries, 0 to 255
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   Date          256 non-null    datetime64[us, UTC]
 1   Holiday_Name  256 non-null    str                
 2   Holiday_Type  256 non-null    category           
 3   _ingested_at  256 non-null    datetime64[us]     
 4   _source_file  256 non-null    str                
dtypes: category(1), datetime64[us, UTC](1), datetime64[us](1), str(2)
memory usage: 20.6 KB


In [62]:
holiday_list['Date'].head()

0   2023-01-06 00:00:00+00:00
1   2023-01-15 00:00:00+00:00
2   2023-01-16 00:00:00+00:00
3   2023-02-03 00:00:00+00:00
4   2023-02-04 00:00:00+00:00
Name: Date, dtype: datetime64[us, UTC]

In [63]:
# outlet coordinates
df = pd.read_parquet('lakehouse/bronze/outlet_coordinates.parquet')
dataset_name = 'outlet_coordinates'
df = check_nulls(df, df.columns, dataset_name)
df = check_duplicates(df, ['Outlet_ID'], dataset_name)
df = check_geo_bounds(df, 'Latitude', 'Longitude', dataset_name)
save_sliver(df, dataset_name)
outlet_cordinates = df
# has 0,0 value we need to drop it
#df.plot(kind='scatter', x='Longitude', y='Latitude')

In [64]:
#outlet_master check

df = pd.read_parquet('lakehouse/bronze/outlet_master.parquet')
dataset_name = 'outlet_master'

df = check_nulls(df, df.columns, dataset_name)

In [65]:
def fix_outlet_type(df, col='Outlet_Type', dataset_name=None, quarantine_unknown=False):
    corrections = {
        'Bakry': 'Bakery',
        'Grocry': 'Grocery',
        ' Eatery': 'Eatery',
        'eatery': 'Eatery',
        'bakery': 'Bakery',
        'grocery': 'Grocery',
        'hotel': 'Hotel',
        'kiosk': 'Kiosk',
        'pharmacy': 'Pharmacy',
        'smmt': 'SMMT',
    }

    valid_types = {'Hotel', 'Grocery', 'SMMT', 'Pharmacy', 'Kiosk', 'Bakery', 'Eatery'}

    df = df.copy()
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(corrections)

    if quarantine_unknown and dataset_name:
        unknown_mask = ~df[col].isin(valid_types)
        if unknown_mask.any():
            qurantine(df[unknown_mask], dataset_name, f"{col}_unknown_value")

        return df[~unknown_mask]
    else:
        return df

In [66]:
df = fix_outlet_type(df, dataset_name=dataset_name)
df = check_duplicates(df, df.columns, dataset_name)
df = check_referential_integrity(df, 'Outlet_ID', outlet_cordinates, 'Outlet_ID', dataset_name)
df = check_format(df, ['Outlet_Type', 'Outlet_Size'], 'category', dataset_name)
df = check_format(df, 'Outlet_ID', 'string', dataset_name)
save_sliver(df, dataset_name)
outlet_master = df
outlet_master.info()

<class 'pandas.DataFrame'>
Index: 19764 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Outlet_ID     19764 non-null  string        
 1   Outlet_Size   19764 non-null  category      
 2   Cooler_Count  19764 non-null  int64         
 3   Outlet_Type   19764 non-null  category      
 4   _ingested_at  19764 non-null  datetime64[us]
 5   _source_file  19764 non-null  str           
dtypes: category(2), datetime64[us](1), int64(1), str(1), string(1)
memory usage: 1.4 MB


In [67]:
# distributer_seasonality
# :TODO fix year month data type
df = pd.read_parquet('lakehouse/bronze/distributor_seasonality_details.parquet')
dataset_name = 'distributor_seasonality_details'
df = check_duplicates(df, ['Distributor_ID', 'Year', 'Month'], dataset_name)
df = check_nulls(df, df.columns, dataset_name)
df = check_format(df, 'Seasonality_Index', 'category', dataset_name)
save_sliver(df, dataset_name)
distributor_seasonality_details = df

In [68]:
df = pd.read_parquet('lakehouse/bronze/transactions_history_final.parquet')
dataset_name = 'transactions_history'
df = check_duplicates(df, ['Outlet_ID', 'Year', 'Month', 'Distributor_ID', 'SKU_ID'], 'transactions')
df = check_nulls(df, df.columns, dataset_name)
df = check_referential_integrity(df, 'Outlet_ID', outlet_cordinates, 'Outlet_ID', dataset_name)
df = check_referential_integrity(df, 'Distributor_ID', distributor_seasonality_details, 'Distributor_ID', dataset_name)
# :TODO! chagne in to a resonable value and fix year month data type
df = check_value_range(df, 'Volume_Liters', 0, np.inf, dataset_name)
df = check_value_range(df, 'Total_Bill_Value', 0, np.inf, dataset_name)
save_sliver(df, dataset_name)
transaction_history = df


In [69]:
transaction_history.info()

<class 'pandas.DataFrame'>
Index: 2334830 entries, 0 to 2376388
Data columns (total 9 columns):
 #   Column            Dtype         
---  ------            -----         
 0   Outlet_ID         str           
 1   Year              int64         
 2   Month             int64         
 3   Distributor_ID    str           
 4   SKU_ID            str           
 5   Volume_Liters     float64       
 6   Total_Bill_Value  float64       
 7   _ingested_at      datetime64[us]
 8   _source_file      str           
dtypes: datetime64[us](1), float64(2), int64(2), str(4)
memory usage: 311.1 MB


In [70]:
holiday_list.info()

<class 'pandas.DataFrame'>
Index: 256 entries, 0 to 348
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   Date          256 non-null    datetime64[us, UTC]
 1   Holiday_Name  256 non-null    str                
 2   Holiday_Type  256 non-null    category           
 3   _ingested_at  256 non-null    datetime64[us]     
 4   _source_file  256 non-null    str                
dtypes: category(1), datetime64[us, UTC](1), datetime64[us](1), str(2)
memory usage: 22.5 KB


In [71]:
# gold layer

In [72]:
#poi part

In [73]:
# !wget https://download.geofabrik.de/asia/sri-lanka-latest.osm.pbf

In [74]:
#!pip install quackosm

In [75]:
INPUT_FILE = "lakehouse/silver/outlet_coordinates.parquet"
OUTPUT_FILE = "lakehouse/gold/poi_features.parquet"
PBF_FILE = "sri-lanka-260515.osm.pbf"
RADIUS_M = 1500

def parse_poi():
    start_time = time.time()

    print("1. Reading map data from local PBF file using QuackOSM...")

    # Define the exact tags we want at the parser level
    tags_filter = {
        'amenity': ['school', 'hospital', 'marketplace'],
        'highway': ['bus_stop'],
        'tourism': ['attraction', 'hotel', 'guest_house'],
        'shop': ['supermarket']
    }

    pois_gdf = qosm.convert_pbf_to_geodataframe(
        PBF_FILE,
        tags_filter=tags_filter
    )
    print(f"-> Extracted {len(pois_gdf)} total POIs from the map.")

    print("\n2. Loading your outlet coordinates...")
    coords = pd.read_parquet(INPUT_FILE)

    outlets_gdf = gpd.GeoDataFrame(
        coords,
        geometry=gpd.points_from_xy(coords['Longitude'], coords['Latitude']),
        crs="EPSG:4326"
    )

    print(f"\n3. Projecting and buffering geometries to {RADIUS_M}m...")


    outlets_metric = outlets_gdf.to_crs("EPSG:3857")
    pois_metric = pois_gdf.to_crs("EPSG:3857")


    outlets_metric['geometry'] = outlets_metric.geometry.buffer(RADIUS_M)

    print("\n4. Performing spatial join... (The fast part!)")
    joined = gpd.sjoin(pois_metric, outlets_metric, how="inner", predicate="intersects")

    print("\n5. Aggregating and formatting results...")
    joined['poi_type'] = None

    if 'amenity' in joined.columns:
        joined.loc[joined['amenity'] == 'school', 'poi_type'] = f'poi_school_{RADIUS_M}m'
        joined.loc[joined['amenity'] == 'hospital', 'poi_type'] = f'poi_hospital_{RADIUS_M}m'
        joined.loc[joined['amenity'] == 'marketplace', 'poi_type'] = f'poi_market_{RADIUS_M}m'

    if 'highway' in joined.columns:
        joined.loc[joined['highway'] == 'bus_stop', 'poi_type'] = f'poi_bus_stop_{RADIUS_M}m'

    if 'shop' in joined.columns:
        joined.loc[joined['shop'] == 'supermarket', 'poi_type'] = f'poi_supermarket_{RADIUS_M}m'

    if 'tourism' in joined.columns:
        joined.loc[joined['tourism'].isin(['attraction', 'hotel', 'guest_house']), 'poi_type'] = f'poi_tourism_{RADIUS_M}m'

    counts = joined.dropna(subset=['poi_type']).groupby(['Outlet_ID', 'poi_type']).size().reset_index(name='count')

    final_df = counts.pivot(index='Outlet_ID', columns='poi_type', values='count').fillna(0).astype(int).reset_index()

    final_df = pd.merge(coords[['Outlet_ID']], final_df, on='Outlet_ID', how='left').fillna(0)

    for col in final_df.columns:
        if col != 'Outlet_ID':
            final_df[col] = final_df[col].astype(int)

    final_df.to_parquet(OUTPUT_FILE, index=False)

    elapsed = time.time() - start_time
    print(f"\nDone! Processed {len(coords)} outlets in {elapsed:.2f} seconds.")
    print(f"Saved to: {OUTPUT_FILE}")

In [76]:
#parse_poi()

In [ ]:
from IPython.utils import io 

# --- Configuration ---
INPUT_FILE = "lakehouse/silver/outlet_coordinates.parquet"
OUTPUT_FILE = "lakehouse/gold/poi_features.parquet"
PBF_FILE = "sri-lanka-260515.osm.pbf"
RADIUS_M = 1500

def parse_poi():
    start_time = time.time()
    
    print("1. Reading map data from local PBF file using QuackOSM...")
    
    tags_filter = {
        'amenity': ['school', 'hospital', 'marketplace'],
        'highway': ['bus_stop'],
        'tourism': ['attraction', 'hotel', 'guest_house'],
        'shop': ['supermarket']
    }
    
    # ---------------------------------------------------------
    # FIX: We swallow QuackOSM's output to prevent the 
    # rich/Jupyter infinite recursion loop.
    # ---------------------------------------------------------
    with io.capture_output() as captured:
        pois_gdf = qosm.convert_pbf_to_geodataframe(
            PBF_FILE,
            tags_filter=tags_filter
        )
        
    print(f"-> Extracted {len(pois_gdf)} total POIs from the map.")

    print("\n2. Loading your outlet coordinates...")
    coords = pd.read_parquet(INPUT_FILE)
    
    outlets_gdf = gpd.GeoDataFrame(
        coords, 
        geometry=gpd.points_from_xy(coords['Longitude'], coords['Latitude']),
        crs="EPSG:4326"
    )

    print(f"\n3. Projecting and buffering geometries to {RADIUS_M}m...")
    outlets_metric = outlets_gdf.to_crs("EPSG:3857")
    pois_metric = pois_gdf.to_crs("EPSG:3857")

    # Draw the 1500m radius circle around every outlet simultaneously
    outlets_metric['geometry'] = outlets_metric.geometry.buffer(RADIUS_M)

    print("\n4. Performing spatial join...")
    joined = gpd.sjoin(pois_metric, outlets_metric, how="inner", predicate="intersects")

    print("\n5. Aggregating and formatting results...")
    joined['poi_type'] = None
    
    if 'amenity' in joined.columns:
        joined.loc[joined['amenity'] == 'school', 'poi_type'] = f'poi_school_{RADIUS_M}m'
        joined.loc[joined['amenity'] == 'hospital', 'poi_type'] = f'poi_hospital_{RADIUS_M}m'
        joined.loc[joined['amenity'] == 'marketplace', 'poi_type'] = f'poi_market_{RADIUS_M}m'
        
    if 'highway' in joined.columns:
        joined.loc[joined['highway'] == 'bus_stop', 'poi_type'] = f'poi_bus_stop_{RADIUS_M}m'
        
    if 'shop' in joined.columns:
        joined.loc[joined['shop'] == 'supermarket', 'poi_type'] = f'poi_supermarket_{RADIUS_M}m'
        
    if 'tourism' in joined.columns:
        joined.loc[joined['tourism'].isin(['attraction', 'hotel', 'guest_house']), 'poi_type'] = f'poi_tourism_{RADIUS_M}m'

    counts = joined.dropna(subset=['poi_type']).groupby(['Outlet_ID', 'poi_type']).size().reset_index(name='count')
    final_df = counts.pivot(index='Outlet_ID', columns='poi_type', values='count').fillna(0).astype(int).reset_index()
    final_df = pd.merge(coords[['Outlet_ID']], final_df, on='Outlet_ID', how='left').fillna(0)
    
    for col in final_df.columns:
        if col != 'Outlet_ID':
            final_df[col] = final_df[col].astype(int)

    final_df.to_parquet(OUTPUT_FILE, index=False)
    
    elapsed = time.time() - start_time
    print(f"\nDone! Processed {len(coords)} outlets in {elapsed:.2f} seconds.")
    print(f"Saved to: {OUTPUT_FILE}")

In [78]:
parse_poi()

1. Reading map data from local PBF file using QuackOSM...


-> Extracted 15925 total POIs from the map.

2. Loading your outlet coordinates...

3. Projecting and buffering geometries to 1500m...

4. Performing spatial join... (The fast part!)

5. Aggregating and formatting results...

Done! Processed 19960 outlets in 18.64 seconds.
Saved to: lakehouse/gold/poi_features.parquet


In [79]:
# Load the generated POI features
poi_features = pd.read_parquet('lakehouse/gold/poi_features.parquet')

print(f"Total outlets with POI data: {len(poi_features)}")
print(poi_features.describe())
print(poi_features.head(10))

Total outlets with POI data: 19960
       poi_bus_stop_1500m  poi_hospital_1500m  poi_market_1500m  \
count        19960.000000        19960.000000      19960.000000   
mean             1.464880            0.394589          0.137625   
std              4.645364            1.421532          0.516327   
min              0.000000            0.000000          0.000000   
25%              0.000000            0.000000          0.000000   
50%              0.000000            0.000000          0.000000   
75%              0.000000            0.000000          0.000000   
max             59.000000           23.000000          8.000000   

       poi_school_1500m  poi_supermarket_1500m  poi_tourism_1500m  
count      19960.000000           19960.000000       19960.000000  
mean           2.019539               0.964329           1.647495  
std            5.586519               2.752137           7.566211  
min            0.000000               0.000000           0.000000  
25%            0.0000

In [80]:
tx = pd.read_parquet('lakehouse/silver/transactions_history.parquet')
outlet_cordinates = pd.read_parquet('lakehouse/silver/outlet_coordinates.parquet')
outlet_master = pd.read_parquet('lakehouse/silver/outlet_master.parquet')
distributor_seasonality_details = pd.read_parquet('lakehouse/silver/distributor_seasonality_details.parquet')
holiday_list = pd.read_parquet('lakehouse/silver/holiday_list.parquet')

In [81]:
# 1. Prepare Tables
tx_gold = pd.read_parquet('lakehouse/silver/transactions_history.parquet')
outlet_cordinates = pd.read_parquet('lakehouse/silver/outlet_coordinates.parquet')
outlet_master = pd.read_parquet('lakehouse/silver/outlet_master.parquet')
distributor_seasonality_details = pd.read_parquet('lakehouse/silver/distributor_seasonality_details.parquet')
holiday_list = pd.read_parquet('lakehouse/silver/holiday_list.parquet')
poi_features = pd.read_parquet('lakehouse/gold/poi_features.parquet')

outlet_full = pd.merge(outlet_master, outlet_cordinates[['Outlet_ID', 'Latitude', 'Longitude']], on='Outlet_ID', how='left')

# 2. Joins: Merge Transactions with Outlets and Seasonality
gold_df = pd.merge(tx_gold, outlet_full.drop(columns=['_ingested_at', '_source_file']), on='Outlet_ID', how='left')
gold_df = pd.merge(gold_df, distributor_seasonality_details.drop(columns=['_ingested_at', '_source_file']), on=['Distributor_ID', 'Year', 'Month'], how='left')

# 3. Feature Engineering: Holiday aggregation
holiday_tmp = holiday_list.copy()
holiday_tmp['Year'] = holiday_tmp['Date'].dt.year
holiday_tmp['Month'] = holiday_tmp['Date'].dt.month
holiday_counts = holiday_tmp.groupby(['Year', 'Month']).size().reset_index(name='Holiday_Count')

# 4. Join Holiday Counts and POI Features
gold_df = pd.merge(gold_df, holiday_counts, on=['Year', 'Month'], how='left')
gold_df['Holiday_Count'] = gold_df['Holiday_Count'].fillna(0).astype(int)

# Merge POI Features (Spatial Context)
gold_df = pd.merge(gold_df, poi_features, on='Outlet_ID', how='left')

# 5. Save to Gold Layer
LAKE_HOUSE_GOLD_PATH = 'lakehouse/gold'
gold_df.to_parquet(f'{LAKE_HOUSE_GOLD_PATH}/master_features.parquet', compression='zstd', index=False)

print('Gold layer master_features with POI context created successfully.')
gold_df.info()
display(gold_df.head())

Gold layer master_features with POI context created successfully.
<class 'pandas.DataFrame'>
RangeIndex: 2334830 entries, 0 to 2334829
Data columns (total 22 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   Outlet_ID              object        
 1   Year                   int64         
 2   Month                  int64         
 3   Distributor_ID         str           
 4   SKU_ID                 str           
 5   Volume_Liters          float64       
 6   Total_Bill_Value       float64       
 7   _ingested_at           datetime64[us]
 8   _source_file           str           
 9   Outlet_Size            category      
 10  Cooler_Count           float64       
 11  Outlet_Type            category      
 12  Latitude               float64       
 13  Longitude              float64       
 14  Seasonality_Index      category      
 15  Holiday_Count          int64         
 16  poi_bus_stop_1500m     int64         
 17  poi_hospit

,Outlet_ID,Year,Month,Distributor_ID,SKU_ID,Volume_Liters,Total_Bill_Value,_ingested_at,_source_file,Outlet_Size,...,Latitude,Longitude,Seasonality_Index,Holiday_Count,poi_bus_stop_1500m,poi_hospital_1500m,poi_market_1500m,poi_school_1500m,poi_supermarket_1500m,poi_tourism_1500m
0,OUT_19886,2024,12,DIST_S_02,SKU_10,5.897879,2177.632359,2026-05-16 18:07:05.434459,data/transactions_history_final.csv,Small,...,6.222572,80.230934,Favorable,7,0,0,0,0,0,0
1,OUT_00837,2024,2,DIST_W_01,SKU_03,20.697364,7244.084814,2026-05-16 18:07:05.434459,data/transactions_history_final.csv,Large,...,7.024894,79.831030,Moderate,10,0,0,0,0,0,0
2,OUT_15438,2025,12,DIST_NW_01,SKU_02,55.101801,13959.108787,2026-05-16 18:07:05.434459,data/transactions_history_final.csv,Medium,...,7.537159,80.103760,Favorable,7,0,0,0,0,0,0
3,OUT_12992,2025,1,DIST_C_01,SKU_07,24.063953,15641.548773,2026-05-16 18:07:05.434459,data/transactions_history_final.csv,Medium,...,7.152607,80.666430,Moderate,7,0,0,0,1,0,0
4,OUT_12334,2025,5,DIST_C_02,SKU_04,47.769665,15525.158656,2026-05-16 18:07:05.434459,data/transactions_history_final.csv,Extra Large,...,6.994638,80.647910,Moderate,10,0,0,0,0,0,0
